## Neural Language Model using RNN and CNN





In [1]:
# Uninstall existing torch and torchtext
!pip uninstall torch torchtext numpy -y

# Install compatible versions
!pip install torch==2.0.1 --index-url https://download.pytorch.org/whl/cu118
!pip install torchtext==0.15.2 numpy==1.24.3
# !pip install numpy

Found existing installation: torch 2.6.0+cu124
Uninstalling torch-2.6.0+cu124:
  Successfully uninstalled torch-2.6.0+cu124
Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 GB 597.5 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.3/63.3 MB 12.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.3/132.3 kB 12.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lit: filename=lit-15.0.7-py3-none-any.whl size=89990 sha256=c7ea2414ddd14aa3e315fe75c16fa43e165b1af6df2655925ef1c74868d36c91
  Stored in directory: /root/.cache/pip/wheels/fc/5d/45/34fe9945d5e45e261134e72284395be36c2d4828af38e2b0fe
Successfully built lit
  Attempting uninstall: triton
    Found existing installation: triton 3.2.0
    Uninstalling triton-3.2.0:
      Successfully uninstalled triton-3.2.0
E

In [17]:
import urllib
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator
import numpy as np



## Corpus and preprocessing

In [19]:

torch.manual_seed(42)


url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
filename = 'shakespeare_data.txt'
urllib.request.urlretrieve(url, filename) ### downloads this text from an external url and puts in the current directory
def read_text_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        return file.read()






text = read_text_file(filename)

tokenizer = get_tokenizer("basic_english")
def yield_tokens(data_iter):
    for text in data_iter:
        yield tokenizer(text)

vocab = build_vocab_from_iterator(yield_tokens([text]), specials=["<unk>"])
vocab.set_default_index(vocab["<unk>"])

def data_process(raw_text_iter):
    data = [torch.tensor([vocab[token] for token in tokenizer(item)], dtype=torch.long) for item in raw_text_iter]
    return torch.cat(tuple(filter(lambda t: t.numel() > 0, data)))

### example
# ["the", "boy", "ran"]
# [3, 12, 7]  (based on vocab)
# tensor([3, 12, 7])



text_pipeline = lambda x: vocab(tokenizer(x))


class LanguageModelDataset(Dataset):
    def __init__(self, data, seq_length):
        self.data = data
        self.seq_length = seq_length

    def __len__(self):
        return len(self.data) - self.seq_length

    def __getitem__(self, index):
        return (self.data[index:index+self.seq_length],
                self.data[index+1:index+self.seq_length+1])







```
data = [1, 2, 3, 4, 5, 6]
seq_length = 3

# index = 0
input  = [1, 2, 3]
target = [2, 3, 4]

# index = 1
input  = [2, 3, 4]
target = [3, 4, 5]

```



In [22]:
def view_tokenized_input_with_tensor(sentence):
    print(f"Input sentence: {sentence}")

    tokens = tokenizer(sentence)
    print(f"Tokens: {tokens}")

    token_ids = [vocab[token] for token in tokens]
    print(f"Token IDs: {token_ids}")

    tensor = torch.tensor(token_ids, dtype=torch.long)
    print(f"Tensor: {tensor}")

print(view_tokenized_input_with_tensor("i love panipuri"))
print(view_tokenized_input_with_tensor("I LOVE panipuri"))

# print(vocab.get_itos()[0])



Input sentence: i love panipuri
Tokens: ['i', 'love', 'panipuri']
Token IDs: [6, 82, 0]
Tensor: tensor([ 6, 82,  0])
None
Input sentence: I LOVE panipuri
Tokens: ['i', 'love', 'panipuri']
Token IDs: [6, 82, 0]
Tensor: tensor([ 6, 82,  0])
None


## Model Definition

In [23]:
class RNNModel(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super(RNNModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        embedded = self.embedding(x)
        output, _ = self.rnn(embedded)
        return self.fc(output)

# class CNNModel(nn.Module):
#     def __init__(self, vocab_size, embed_size, num_filters, filter_sizes):
#         super(CNNModel, self).__init__()
#         self.embedding = nn.Embedding(vocab_size, embed_size)
#         self.convs = nn.ModuleList([
#             nn.Conv2d(1, num_filters, (fs, embed_size)) for fs in filter_sizes
#         ])
#         self.fc = nn.Linear(len(filter_sizes) * num_filters, vocab_size)

#     def forward(self, x):
#         embedded = self.embedding(x).unsqueeze(1)
#         conved = [nn.functional.relu(conv(embedded)).squeeze(3) for conv in self.convs]
#         pooled = [nn.functional.max_pool1d(conv, conv.shape[2]).squeeze(2) for conv in conved]
#         cat = torch.cat(pooled, dim=1)
#         return self.fc(cat)



In [25]:
vocab_size, embed_size, hidden_size =  8, 16, 32


In [27]:
nn.LSTM(embed_size, hidden_size, batch_first=True)

LSTM(16, 32, batch_first=True)

## Training and evaluation function

In [9]:
def train(model, data_loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for batch, (inputs, targets) in enumerate(data_loader):
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        output = model(inputs)
        loss = criterion(output.view(-1, len(vocab)), targets.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(data_loader)




def evaluate(model, data_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for inputs, targets in data_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            output = model(inputs)
            loss = criterion(output.view(-1, len(vocab)), targets.view(-1))
            total_loss += loss.item()
    return total_loss / len(data_loader)



## Helper functions for sequence probability, perplexity adn text generation

In [10]:
# 1. Probability of a sequence of words
def sequence_probability(model, sequence, device):
    model.eval()
    tokens = torch.tensor(text_pipeline(sequence)).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(tokens)
        probs = torch.nn.functional.softmax(output, dim=-1)
    return probs[0, -1, tokens[0, -1]].item() ## probs -> (1, sequence_length, vocab_size)
                                              ## tokens -> (1, sequence_length)


# 2. Perplexity computation of a sequence
def compute_perplexity(model, sequence, device):
    model.eval()
    tokens = torch.tensor(text_pipeline(sequence)).unsqueeze(0).to(device)
    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        output = model(tokens)
        loss = criterion(output.view(-1, len(vocab)), tokens.view(-1))

    return torch.exp(loss).item()


# 3. Text generation
def generate_text(model, seed_sequence, max_length, device):
    model.eval()
    tokens = torch.tensor(text_pipeline(seed_sequence)).unsqueeze(0).to(device)
    generated = list(tokens[0].cpu().numpy())

    with torch.no_grad():
        for _ in range(max_length):
            output = model(tokens)
            probs = torch.nn.functional.softmax(output[:, -1], dim=-1)
            next_token = torch.multinomial(probs, num_samples=1).item()
            generated.append(next_token)
            tokens = torch.cat([tokens, torch.tensor([[next_token]]).to(device)], dim=1)

    return " ".join([vocab.get_itos()[idx] for idx in generated])


## Hyperparameters

In [28]:

EMBED_SIZE = 16 # 128
HIDDEN_SIZE = 8 # 256
NUM_FILTERS = 100 ## for CNN
FILTER_SIZES = [3, 4, 5] ## for CNN
BATCH_SIZE = 32
NUM_EPOCHS = 1 # 20
LEARNING_RATE = 0.1
SEQ_LENGTH = 20
TRAIN_SPLIT = 0.8



## Training


In [16]:

data = data_process([text])
dataset = LanguageModelDataset(data, SEQ_LENGTH)

# split the dataset into training and vadation sets
train_size = int(TRAIN_SPLIT * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# sreate the train and val data loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
rnn_model = RNNModel(len(vocab), EMBED_SIZE, HIDDEN_SIZE).to(device)
# cnn_model = CNNModel(len(vocab), EMBED_SIZE, NUM_FILTERS, FILTER_SIZES).to(device)



print(f"Training started...")
optimizer = optim.Adam(rnn_model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()

for epoch in range(NUM_EPOCHS):
    train_loss = train(rnn_model, train_loader, optimizer, criterion, device)
    val_loss = evaluate(rnn_model, val_loader, criterion, device)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")



Training started...
Epoch 1/1, Train Loss: 5.6607, Val Loss: 5.5912


In [29]:
rnn_model = RNNModel(len(vocab), EMBED_SIZE, HIDDEN_SIZE)

In [30]:
print(rnn_model)

RNNModel(
  (embedding): Embedding(12236, 16)
  (rnn): LSTM(16, 8, batch_first=True)
  (fc): Linear(in_features=8, out_features=12236, bias=True)
)


In [32]:
 # 1. Probability of a sequence of words



seq = "you know"
prob = sequence_probability(rnn_model, seq, "cpu")
print(f"Probability of '{seq}': {prob:.6f}")

# 2. Perplexity computation
perplexity = compute_perplexity(rnn_model, seq, "cpu")
print(f"Perplexity: {perplexity:.2f}")

# 3. Text generation
seed = "the boy was"
generated = generate_text(rnn_model, seed, max_length=20, device="cpu")
print(f"Generated text: {generated}")


Probability of 'you know': 0.000081
Perplexity: 10938.20
Generated text: the boy was distemper uncomfortable stifled prefix un religion shade deflower banquet oppression salute doom dunsmore mars amount strangling heaviness spokest requires famed


## Resources

1.   Tensors with Pytorch -> https://pytorch.org/tutorials/beginner/deep_learning_60min_blitz.html
2.   https://pytorch.org/tutorials/intermediate/char_rnn_generation_tutorial.html
3.   https://pytorch.org/tutorials/intermediate/char_rnn_classification_tutorial.html
4.  LSTM: https://karpathy.github.io/2015/05/21/rnn-effectiveness/

